# Text, Binary, and Temporary Files

## Handling File Encodings

In [1]:
import os

# Globomantics – Multilingual Customer Review Handler
# Simulated customer reviews from multiple regions (including special characters)
customer_reviews = [
    "Great service and fast delivery!", # English (UTF-8)
    "Servicio excelente y entrega rápida.", # Spanish (UTF-8)
    "Livraison rapide et efficace. Très satisfait.", # French (UTF-8)
    "Lieferung war schnell. Sehr zufrieden!", # German (UTF-8)
    "Entrega ótima. Estou muito satisfeito!", # Portuguese (UTF-8)
    "Доставка была очень быстрой. Спасибо!", # Russian (UTF-8)
    "配送がとても早かったです。ありがとうございました！", # Japanese (UTF-8)
]

# Folder to store the reviews
folder = "reviews"
if not os.path.exists(folder):
    os.makedirs(folder)

# File path for saving the reviews
utf8_file = f"{folder}/customer_reviews_utf8.txt"
iso_file = f"{folder}/customer_reviews_iso8859_1.txt"

with open(utf8_file, 'w', encoding='utf-8') as file:
    for review in customer_reviews:
        file.write(review + "\n")

print(f"Saved reviews using UTF-8 encoding: {utf8_file}")

print("\ntrying to read UTF-8 file using  incorrect encoding (ISO-8859-1): ")
try:
    with open(utf8_file, 'r', encoding='iso-8859-1') as file:
        contents = file.read()
        print(contents)
except UnicodeDecodeError as e:
    print(f"Decode Error: {e}")

filtered_reviews = [r for r in customer_reviews if all(ord(c) < 256 for c in r)]

with open(iso_file, 'w', encoding='iso_8859-1') as file:
    for review in filtered_reviews:
        file.write(review + "\n")

print(f"\nSaved ISO-8859-1-compatible reviews to: {iso_file}")

Saved reviews using UTF-8 encoding: reviews/customer_reviews_utf8.txt

trying to read UTF-8 file using  incorrect encoding (ISO-8859-1): 
Great service and fast delivery!
Servicio excelente y entrega rÃ¡pida.
Livraison rapide et efficace. TrÃ¨s satisfait.
Lieferung war schnell. Sehr zufrieden!
Entrega Ã³tima. Estou muito satisfeito!
ÐÐ¾ÑÑÐ°Ð²ÐºÐ° Ð±ÑÐ»Ð° Ð¾ÑÐµÐ½Ñ Ð±ÑÑÑÑÐ¾Ð¹. Ð¡Ð¿Ð°ÑÐ¸Ð±Ð¾!
ééãã¨ã¦ãæ©ãã£ãã§ãããããã¨ããããã¾ããï¼


Saved ISO-8859-1-compatible reviews to: reviews/customer_reviews_iso8859_1.txt


### Detecting file encoding using chardet

In [2]:
# pip install chardet

In [3]:
import chardet

files_to_check = [utf8_file, iso_file]

for filepath in files_to_check:
    with open(filepath, 'rb') as f:
        raw = f.read()
    result = chardet.detect(raw)
    print(f"{filepath}: {result}")

# Fallback: probe common encodings manually
def detect_encoding(filepath):
    for enc in ['utf-8', 'iso-8859-1', 'cp1252', 'utf-16']:
        try:
            with open(filepath, encoding=enc) as f:
                f.read()
            return enc
        except UnicodeDecodeError:
            continue
    return 'unknown'

print()
for filepath in files_to_check:
    enc = detect_encoding(filepath)
    print(f"{filepath} → first compatible encoding: {enc}")

reviews/customer_reviews_utf8.txt: {'encoding': 'utf-8', 'confidence': 0.99, 'language': 'ja', 'mime_type': 'text/plain'}
reviews/customer_reviews_iso8859_1.txt: {'encoding': 'Windows-1252', 'confidence': 0.21999419052532126, 'language': 'de', 'mime_type': 'text/plain'}

reviews/customer_reviews_utf8.txt → first compatible encoding: utf-8
reviews/customer_reviews_iso8859_1.txt → first compatible encoding: iso-8859-1


## Working with Binary Files

In [4]:
# Globomantics – Binary File Archiver for Uploaded Shipping Docs
# Source files (simulating uploaded content)
pdf_source = "uploads/sample_invoice.pdf"
image_source = "uploads/sample_delivery.jpg"

# Target archive folder
archive_folder = "archived_docs"

# Ensure directories exist
os.makedirs("uploads", exist_ok=True)
os.makedirs(archive_folder, exist_ok=True)

# File destinations
pdf_target = os.path.join(archive_folder, "2025-07-24_invoice_copy.pdf")
image_target = os.path.join(archive_folder, "2025-07-24_delivery_photo.jpg")

pdf_in = open(pdf_source, 'rb')
pdf_bytes = pdf_in.read()
pdf_in.close()

pdf_out = open(pdf_target, 'wb')
pdf_out.write(pdf_bytes)
pdf_out.close()

print(f"PDF archived {pdf_target} (Size: {len(pdf_bytes)} bytes)")

img_in = open(image_source, 'rb')
img_bytes = img_in.read()
img_in.close()

img_out = open(image_target, 'wb')
img_out.write(img_bytes)
img_out.close()

print(f"Image archived: {image_target} (Size: {len(img_bytes)} bytes)")

PDF archived archived_docs/2025-07-24_invoice_copy.pdf (Size: 13480 bytes)
Image archived: archived_docs/2025-07-24_delivery_photo.jpg (Size: 199152 bytes)


## Temporary and In-memoy Files

In [6]:
import tempfile

# Globomantics – In-Memory & Temporary File Handler
# === Part 1: Using StringIO to simulate a text file upload (e.g., from a web form) ===

from io import StringIO

uploaded_text = """Client ID: 45231
Region: East Hub
Notes: Package delayed due to weather
Resolution: Priority delivery requested"""

text_file = StringIO(uploaded_text)
print("=== Reading Uploaded Text File from Memory ===")
for line in text_file:
    print(line.strip())

text_file.close()

# === Part 2: Using BytesIO to simulate a binary file upload (e.g., image or PDF) ===
from io import BytesIO

pdf_binary_data = b"%PDF-1.4\nDummy PDF File Header...\n..."

pdf_file = BytesIO(pdf_binary_data)
print("\n=== Reading Uploaded Binary File from Memory")
header = pdf_file.read(15)
print(f"Binary file header: {header}")

pdf_file.close()

# === Part 3: Using tempfile to create a temporary directory and write a review file ===

print("\n=== Writing Temporary Review File ===")
with tempfile.TemporaryDirectory() as tmp_dir:
    temp_path = os.path.join(tmp_dir, "client_review.txt")
    with open(temp_path, 'w') as temp_file:
        temp_file.write("Client: GLOBO123456\nRating: 4/5\nComment: Smooth process overall.")
    print(f"Temporary file created at: {temp_path}")

    with open(temp_path, 'r') as temp_file:
        print("Contents of temp file:")
        print(temp_file.read())

print("\nTemporary directory and file removed after block exists.")

=== Reading Uploaded Text File from Memory ===
Client ID: 45231
Region: East Hub
Notes: Package delayed due to weather
Resolution: Priority delivery requested

=== Reading Uploaded Binary File from Memory
Binary file header: b'%PDF-1.4\nDummy '

=== Writing Temporary Review File ===
Temporary file created at: /var/folders/l3/6dn78ndx5cb9nbc_4r6nycq40000gn/T/tmpy2wig7gv/client_review.txt
Contents of temp file:
Client: GLOBO123456
Rating: 4/5
Comment: Smooth process overall.

Temporary directory and file removed after block exists.
